# 🛡️ SAFIR — Uctan Uca (End-to-End) Teknik Demonstrasyon

Bu defter, SAFIR'in **gercek calisan sisteminin teknik aynasidir**. Production `SafirPipeline`'in **gercek asama metotlarini tek tek cagirir** ve her cagrinin **gercek** ciktisini gosterir.

> Notebook production mantigini **kopyalamaz** ve her seyi tek `run()` arkasina **saklamaz** — orchestration + observability katmanidir. VLM olarak **Gemini** kullanilir.

### Gercek pipeline (koddan cikarilmis) ve cagrilan metotlar
```
VIDEO
  ↓  SafirPipeline.stage_sample()   -> AdaptiveFrameSampler.process_video (kumeleme YOK, tum kanit kareleri)
  ↓  SafirPipeline.stage_vlm()      -> GeminiVLM.analyze_evidence_batched + reconcile_events → GEMINI
  ↓  SafirPipeline.stage_events()   -> EventEngine.detect + TemporalReasoner.reason + RuleEngine.evaluate
  ↓  SafirPipeline.stage_context()  -> ContextBuilder.build (SQLite + FAISS RAG)
  ↓  SafirPipeline.stage_decide()   -> SafirAgent.run (LangGraph)
  ↓  SafirPipeline.stage_escalate() -> EscalationPolicy.evaluate
  ↓  SafirPipeline.build_report()   -> SafirReport (+to_sartname_json)
```
Ayni metotlar `SafirPipeline.run()` tarafindan da ayni sirayla cagrilir (tek gercek kaynak).

## 1) Environment & Configuration
Proje koku, konfigurasyon, cihaz, model bilgileri. Eksik bagimlilik varsa **acik** hata verilir.

> `DEV_MODE` yalnizca gelistirici dogrulamasi icindir (mock VLM/LLM + sahte RAG, offline). **Mentor demosu icin `DEV_MODE=False`** (gercek Gemini + gercek RAG).

In [ ]:
import sys, os, platform
from pathlib import Path
_here = Path.cwd()
PROJECT_ROOT = _here if (_here / 'src').exists() else _here.parent
assert (PROJECT_ROOT / 'src').exists(), 'Proje koku (safir-ai/) bulunamadi.'
sys.path.insert(0, str(PROJECT_ROOT)); os.chdir(PROJECT_ROOT)

# ---- MOD ----
DEV_MODE = True   # False: GERCEK (Gemini + gercek RAG) | True: offline dogrulama (mock+fake)

_missing = []
for _m in ['cv2', 'numpy', 'langgraph', 'langchain_openai', 'httpx', 'yaml', 'ipywidgets']:
    try: __import__(_m)
    except Exception as e: _missing.append(f'{_m} ({e})')
if _missing:
    raise ImportError('Eksik bagimlilik: ' + ', '.join(_missing) + '  ->  pip install -r requirements-gemini.txt')

from src.utils.config_loader import load_config
config = load_config()
if DEV_MODE:
    config = config.model_copy(update={'app': config.app.model_copy(update={'use_mock_vlm': True, 'use_mock_llm': True})})

print('Python      :', platform.python_version())
print('Device (cfg):', config.system.device)
print('VLM backend :', config.vlm.active_model, '->', config.vlm.models[config.vlm.active_model].model_name)
print('LLM backend :', config.llm.active_model, '->', config.llm.models[config.llm.active_model].model_name)
print('DEV_MODE    :', DEV_MODE, '(False = gercek Gemini + gercek RAG)')
if not DEV_MODE and not os.environ.get('GEMINI_API_KEY'):
    raise EnvironmentError('GEMINI_API_KEY tanimli degil. Gercek demo icin sart. (Secret: ortam degiskeni; koda yazma.)')

## 2) Input Video (gercek video)
Videoyu `data/` klasorune koy ve **acilir listeden sec** (en guvenilir). Sonra metadata + ornek kareler gorunur.
> Sentetik video yalnizca hicbir video secilmediginde, acikca isaretli bir **fallback**tir; mentor demosunda gercek video kullanin.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
Path('data').mkdir(exist_ok=True)
_exts = ('*.mp4', '*.avi', '*.mkv', '*.mov')
_vids = sorted({str(p) for e in _exts for p in Path('data').glob(e)})
video_dd = widgets.Dropdown(options=['(sentetik-fallback)'] + _vids, value=(_vids[0] if _vids else '(sentetik-fallback)'),
                            description='data/ video:', style={'description_width': 'initial'}, layout=widgets.Layout(width='560px'))
path_box = widgets.Text(value='', placeholder='veya tam yol', description='Yol:', layout=widgets.Layout(width='560px'))
display(widgets.VBox([video_dd, path_box]))
print(f'data/ altinda {len(_vids)} video bulundu. Sec -> ALT hucreyi calistir.')
print('(data/ye yeni video eklediysen bu hucreyi TEKRAR calistir.)')

In [ ]:
import tempfile, cv2, numpy as np
def _resolve_video():
    if video_dd.value and video_dd.value != '(sentetik-fallback)' and Path(video_dd.value).exists():
        print('Kaynak: dropdown'); return video_dd.value
    if path_box.value and Path(path_box.value).exists():
        print('Kaynak: yol kutusu'); return path_box.value
    tmp = tempfile.mkdtemp(); p = str(Path(tmp) / 'synthetic.mp4')
    frames = [np.full((240, 320, 3), 30, np.uint8) for _ in range(75)]
    for i in range(25, 50): cv2.rectangle(frames[i], (40, 60), (260, 190), (200, 200, 200), -1)
    w = cv2.VideoWriter(p, cv2.VideoWriter_fourcc(*'mp4v'), 25.0, (320, 240))
    for f in frames: w.write(f)
    w.release(); print('[!] Gercek video secilmedi -> SENTETIK fallback (mentor demosunda gercek video sec).'); return p

VIDEO_PATH = _resolve_video()
cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS) or 0; N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
DURATION = N_FRAMES / FPS if FPS else 0
print(f'Video       : {VIDEO_PATH}')
print(f'Cozunurluk  : {W}x{H}\nFPS         : {FPS:.1f}\nSure        : {DURATION:.1f} s\nKare sayisi : {N_FRAMES}')
row = []
for idx in [0, N_FRAMES // 2, max(0, N_FRAMES - 1)]:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx); ok, fr = cap.read()
    if ok: _ok, _b = cv2.imencode('.jpg', fr); row.append(widgets.Image(value=_b.tobytes(), format='jpeg', width=180))
cap.release(); print('Ornek kareler (bas/orta/son):'); display(widgets.HBox(row))

## 3) Production Pipeline'i Kur (gercek __init__ wiring)
`SafirPipeline(config)` tum gercek bilesenleri (VLM/Gemini, EventEngine, TemporalReasoner, RuleEngine, ContextBuilder, SafirAgent, EscalationPolicy, EventStore, RAG) production'daki gibi kurar. Asagida bu orneginin **gercek asama metotlarini** tek tek cagiracagiz.

In [ ]:
if DEV_MODE:
    # Sadece DEV: gercek Gemini embedding/rerank API'lerini cagirmamak icin RAG'i hafif sahteyle degistir.
    from dataclasses import dataclass
    import src.main as safir_main
    @dataclass
    class _Doc:
        text: str; score: float = 1.0
    class _FakeRAG:
        def seed_default_regulations(self): pass
        def query(self, q, top_k=None):
            return [_Doc('ISG Yonetmeligi Madde 24: KKD zorunludur.'), _Doc('Operasyonel Kural OK-07: yaya gecidi.')][:(top_k or 2)]
    safir_main.EmbeddingRAGService = lambda *a, **k: _FakeRAG()
else:
    import src.main as safir_main  # gercek EmbeddingRAGService (Gemini API) — GEMINI_API_KEY gerektirir

USER_PROMPT = 'Sahnede riskli bir durum var mi degerlendir.'
pipeline = safir_main.SafirPipeline(config)
print('SafirPipeline hazir. VLM =', type(pipeline._vlm).__name__, '| Agent LLM =', pipeline._agent.model_name)

In [ ]:
# Gorsel yardimcilari
import base64
def _img(b, width=180): return widgets.Image(value=b, format='jpeg', width=width)
def _b64(u): return base64.b64decode(u.split(',', 1)[1])
def _draw_bbox(image_bytes, bbox):
    arr = cv2.imdecode(np.frombuffer(image_bytes, np.uint8), cv2.IMREAD_COLOR)
    if bbox:
        x0, y0, x1, y1 = bbox; cv2.rectangle(arr, (int(x0), int(y0)), (int(x1), int(y1)), (0, 0, 255), 2)
    _ok, buf = cv2.imencode('.jpg', arr); return buf.tobytes()

## 4) `pipeline.stage_sample()` → Frame Sampling (kumeleme YOK)
**INPUT:** video · **PROCESSING:** gercek `AdaptiveFrameSampler` (CPU) · **OUTPUT:** esigi gecen TUM kanit kareleri + `motion_bbox` (olay kumelemesi burada YAPILMAZ, VLM'in isidir).

In [ ]:
sampler, evidence = pipeline.stage_sample(VIDEO_PATH)
st = sampler.last_run_stats
ratio = (100.0 * st.sampled_frames_evaluated / st.total_frames_scanned) if st.total_frames_scanned else 0
print(f'Original frames : {st.total_frames_scanned}')
print(f'Sampled frames  : {st.sampled_frames_evaluated}  (%{ratio:.1f})')
print(f'Evidence frames : {st.evidence_frame_count}  (elenen %{st.eliminated_ratio_pct})')
print('\nKanit kareleri (kirmizi kutu = motion_bbox, hicbiri konumsal olarak etiketlenmez):')
display(widgets.HBox([_img(_draw_bbox(f.image_bytes, f.motion_bbox)) for f in evidence]))

## 5) `pipeline.stage_vlm()` → VLM (Gemini) Girdi & Ham Yanit
**INPUT (Gemini'ye giden):** TUM kanit kareleri + prompt · **PROCESSING:** gercek `GeminiVLM.analyze_evidence_batched` (kronolojik batch'ler) + gerekirse `reconcile_events` · **OUTPUT:** modelin **ham** yaniti + VLM'in KENDI urettigi olay kumeleri (`EVENTS_JSON`).

In [ ]:
from src.prompts import VLM_OBSERVER_SYSTEM_PROMPT
print('=== GEMINI INPUT ===')
print('Kullanici istemi:', USER_PROMPT)
print('Sistem istemi (ilk 200 krk):', VLM_OBSERVER_SYSTEM_PROMPT[:200], '...')
print('Gonderilen kare sayisi:', len(evidence))

vlm_response = pipeline.stage_vlm(evidence, USER_PROMPT)   # <-- gercek Gemini cagrisi (batching + reconciliation)

print('\n=== GEMINI RAW OUTPUT (model:', vlm_response.model_name, ') ===\n')
print(vlm_response.description)
print('\n=== Parse edilmis EVENTS_JSON (VLM tarafindan kumelenmis olaylar) ===')
for e in vlm_response.structured_events: print('  ', e)
if vlm_response.description.startswith('[HATA]'):
    print('\n[!] Gemini cagrisi basarisiz -> pipeline dayanikliligi devrede (degraded).')

## 6) `pipeline.stage_events()` → Event / Temporal Analysis
**PROCESSING:** gercek `EventEngine.detect` → `TemporalReasoner.reason` → `RuleEngine.evaluate` · **OUTPUT:** tipli olaylar, zamansal sureklilik, tetiklenen ISG kurallari.

In [ ]:
detected, temporal, rules, latest_ts = pipeline.stage_events(vlm_response, evidence)
print('Tespit edilen olaylar (DetectedEvent):')
for d in detected:
    print(f'   {{"event": "{d.event_type}", "time": {d.timestamp:.1f}, "confidence": {d.confidence:.2f}}}')
print('\nZamansal olaylar (TemporalEvent):')
for t in temporal: print(f'   {t.event_type:<22} tekrar={t.occurrence_count} sure={t.duration:.1f}s')
print('\nTetiklenen ISG kurallari (RuleMatch):')
for r in rules: print(f'   [{r.rule_id}] ({r.severity}) {r.rule_description}')

## 7) `pipeline.stage_context()` → Ajan Baglami (RAG dahil)
**PROCESSING:** gercek `ContextBuilder.build` (SQLite + FAISS RAG) + kural ozeti · **OUTPUT:** ajana gidecek istem blogu + ilgili mevzuat.

In [ ]:
prompt_block, context = pipeline.stage_context(vlm_response, USER_PROMPT, latest_ts, rules)
print('Ilgili mevzuat (RAG):')
for r in context.relevant_regulations: print('   -', r)
print('\nAjana giden istem blogu (ilk 600 krk):\n')
print(prompt_block[:600], '...')

## 8) `pipeline.stage_decide()` + `stage_escalate()` → Karar / Risk
**PROCESSING:** gercek `SafirAgent.run` (LangGraph) → `EscalationPolicy.evaluate` · **OUTPUT:** risk seviyesi, ozet, aksiyonlar, otomatik eskalasyon.

In [ ]:
decision = pipeline.stage_decide(prompt_block)     # <-- gercek ajan (Gemini LLM)
escalation = pipeline.stage_escalate(decision, vlm_response)
print('Risk Level :', decision.risk_level.upper(), f'(skor {decision.risk_score}/100)')
print('Ozet       :', decision.summary)
print('Aksiyonlar :')
for a in decision.actions: print('   -', a)
print('\nOtomatik eskalasyon:', escalation.tier.value, '| alarm otomatik tetiklendi:', escalation.auto_dispatched)
print('Gerekce            :', escalation.reason)

## 9) `pipeline.build_report()` → Final Report
**PROCESSING:** gercek olay kaydi (EventBuilder/History/Store) + `SafirReport` · **OUTPUT:** sartname-uyumlu JSON + insan-okur ozet.

In [ ]:
import json
report = pipeline.build_report(
    video_source=VIDEO_PATH, sampler=sampler, evidence_frames=evidence, vlm_response=vlm_response,
    context=context, decision=decision, escalation=escalation,
    temporal_events=temporal, rule_matches=rules, latest_timestamp=latest_ts)

print('===== JSON (sartname uyumlu) =====')
print(json.dumps(report.to_sartname_json(), ensure_ascii=False, indent=2))
print('\n===== Insan-okur ozet =====')
print('Video Summary      :', report.summary or report.natural_language_summary)
print('Detected Events    :', report.detected_event_types)
print('Critical Moments   :', [f'{e.timestamp:.1f}s: {e.description[:48]}' for e in report.timeline])
print('Risk Level         :', report.risk_level.upper(), f'({report.risk_score}/100)')
print('Recommended Actions:')
for a in (report.actions or [report.recommended_action]): print('   -', a)

## 10) SAFIR END-TO-END TEST

In [ ]:
gemini_ok = not vlm_response.description.startswith('[HATA]')
checks = [
    ('Video Input',      Path(VIDEO_PATH).exists()),
    ('Frame Sampling',   st.evidence_frame_count > 0),
    ('VLM Event Cluster',len(vlm_response.structured_events) > 0),
    ('Gemini Inference', gemini_ok),
    ('Output Parsing',   isinstance(vlm_response.structured_events, list)),
    ('Event Analysis',   len(detected) > 0),
    ('Risk/Decision',    decision is not None),
    ('Escalation',       escalation is not None),
    ('Structured JSON',  bool(report.to_sartname_json())),
    ('Final Report',     report.event_id is not None),
]
print('=' * 42); print('        SAFIR END-TO-END TEST'); print('=' * 42)
for name, ok in checks:
    mark = '✓' if ok else '✗'
    print(f'  {name:<22} {mark}')
print('=' * 42)
overall = all(ok for _, ok in checks)
print('STATUS:', 'PASS' if overall else 'BLOCKED')
if not overall:
    failed = [n for n, ok in checks if not ok][0]
    print('\nFailed stage :', failed)
    if failed == 'Gemini Inference':
        print('Error        :', vlm_response.description)
        print('Root cause   : Gemini cagrisi basarisiz (kota/model/anahtar).')
        print('Cozum        : GEMINI_API_KEY + config.yaml model_name (kotali model).')